In [ ]:
# Run this cell to install ChromaDB if desired
try:
    assert version('chromadb') == '0.4.17'
except:
    !pip install chromadb==0.4.17
try:
    assert version('pysqlite3') == '0.5.2'
except:
    !pip install pysqlite3-binary==0.5.2
__import__('pysqlite3')
import sys
sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')
import chromadb

In [ ]:
# Load the dataset
import pandas as pd
reviews = pd.read_csv("womens_clothing_e-commerce_reviews.csv")

# Display the first few entries
reviews.head()

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
from openai import OpenAI
from sklearn.manifold import TSNE
from scipy.spatial import distance

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def create_embeddings(texts):  
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts)
    response_dict = response.model_dump()
    return [data['embedding'] for data in response_dict['data']]
    
def create_review_text(review):
    return f"""Review ID:{review["Review ID"]} Clothing ID:{review["Clothing ID"]} Age: {review["Age"]} Title: {review["Title"]} Review Text: {review["Review Text"]} Rating: {review["Rating"]} Recommended IND: {review["Recommended IND"]} Positive Feedback Count: {review["Positive Feedback Count"]} Division Name: {review["Division Name"]} Department Name: {review["Department Name"]} Class Name: {review["Class Name"]} """


review_texts = [create_review_text(row) for index, row in reviews.iterrows()]  
embeddings = create_embeddings(review_texts)


tsne = TSNE(n_components=2, perplexity=5)
embeddings_2d = tsne.fit_transform(np.array(embeddings))

plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1])


plt.show()


def find_n_closest(query_vector, embeddings, n=3):
  distances = []
  for index, embedding in enumerate(embeddings):
    # Calculate the cosine distance between the query vector and embedding
    dist = distance.cosine(query_vector,embedding)
    # Append the distance and index to distances
    distances.append({"distance": dist, "index": index})
  # Sort distances by the distance key
  distances_sorted = sorted(distances,key = lambda x:x['distance'])
  # Return the first n elements in distances_sorted
  return distances_sorted[0:n]
 
def find_closest(query_vector, embeddings):
  distances = []
  for index, embedding in enumerate(embeddings):
    dist = distance.cosine(query_vector, embedding)
    distances.append({"distance": dist, "index": index})
  return min(distances, key=lambda x: x['distance'])

query_text = "quality, fit, style, comfort "   
query_vector = create_embeddings(query_text)[0]

hits = find_n_closest(query_vector,embeddings,n=5)

#print(f'Search results for "{query_text}"')
for hit in hits:
  # Extract the product at each index in hits
  review = reviews.iloc[hit['index']]
  print(review["Review Text"])

clientchroma = chromadb.PersistentClient()

"""collection = clientchroma.create_collection(
  name="cloth_reviews",
  embedding_function=OpenAIEmbeddingFunction(model_name="text-embedding-3-small", 
                                            # api_key="<OPENAI_API_TOKEN>")
#)

collection = clientchroma.get_or_create_collection(
  name="cloth_reviews",
  embedding_function=OpenAIEmbeddingFunction(model_name="text-embedding-3-small", 
                                            # api_key="<OPENAI_API_TOKEN>")"""
collection = clientchroma.get_collection(
  name="cloth_reviews",
  embedding_function=OpenAIEmbeddingFunction(model_name="text-embedding-3-small"))
ids = []
documents = []

# For loop: just append id and review_text
for index, row in reviews.iterrows():
    ids.append(str(row['Review ID']))
    documents.append(str(row['Review Text'] or ""))  # handles NaN

# Outside loop: ONE add
collection.add(
    ids=ids,
    documents=documents
)

dic_most_similar_reviews = collection.query(
    query_texts=["Absolutely wonderful - silky and sexy and comfortables"],
    n_results=3
)


nested = dic_most_similar_reviews['documents']
most_similar_reviews = [item for sublist in nested for item in sublist]    
print("\n\n\n three most similar reviews:")
print(most_similar_reviews)

print("\n\n")
print("=== Collection metadata ===")
print("Name :", collection.name)
print("Count:", collection.count())                     # how many docs are stored
print("ID   :", collection.id)

